# Выгрузка фич 2018-2023 в 0.1° (с половинами для band1)

**Стратегия выгрузки:**
- band 0: 30-80°E — полная полоса (8.5k точек)
- band 1: 80-130°E — **разбита на две половины** (по 5.7k каждая)
  - band 1 west: 80-105°E
  - band 1 east: 105-130°E
- band 2: 130-180°E — полная полоса (8.5k точек)

Итого: **4 задачи на год × 6 лет = 24 задачи** в очереди GEE.

**Чиненные баги:**
1. `get_lst_features` — фильтрация `calendarRange` ДО `.map` (timestamps на месте)
2. `get_landsat` — для 2021+ объединение Landsat 8 + 9 (больше снимков)

**Куда смотреть:**
- статус задач: https://code.earthengine.google.com/tasks
- результат в Drive: `GEE_exports_features_01deg/`

In [ ]:
import ee
ee.Authenticate()
ee.Initialize(project='clean-outcome-446214-a4')
print('GEE OK')

## Параметры

In [ ]:
STEP_DEG = 0.1
EXPORT_FOLDER = 'GEE_exports_features_01deg'
YEARS = [2018, 2019, 2020, 2021, 2022, 2023]
SOUTH, NORTH = 55, 78

# 4 поддиапазона по долготе: band0, band1_west, band1_east, band2
BANDS = [
    ('band0',      30, 80),
    ('band1_west', 80, 105),
    ('band1_east', 105, 130),
    ('band2',      130, 180),
]

print(f'Шаг: {STEP_DEG}° (~{STEP_DEG*111:.0f} км)')
print(f'Годы: {YEARS}')
print(f'Поддиапазонов: {len(BANDS)}')
print(f'Всего задач: {len(YEARS)} × {len(BANDS)} = {len(YEARS)*len(BANDS)}')

## Функции для признаков (с исправлениями)

In [ ]:
def get_landsat(year, region):
    """Летние индексы. Landsat 8 для всех годов, плюс Landsat 9 для 2021+."""
    l8 = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
          .filterDate(f'{year}-06-01', f'{year}-08-31')
          .filterBounds(region)
          .filter(ee.Filter.lt('CLOUD_COVER', 60)))
    if year >= 2021:
        l9 = (ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')
              .filterDate(f'{year}-06-01', f'{year}-08-31')
              .filterBounds(region)
              .filter(ee.Filter.lt('CLOUD_COVER', 60)))
        coll = l8.merge(l9)
    else:
        coll = l8
    # L8/L9 каналы (Collection 2 SR): B4=RED, B5=NIR, B3=GREEN, B6=SWIR
    def renamebands(img):
        return img.select(['SR_B4', 'SR_B5', 'SR_B3', 'SR_B6'],
                          ['RED', 'NIR', 'GREEN', 'SWIR']).multiply(2.75e-5).add(-0.2)
    img = coll.map(renamebands).median()
    ndvi = img.expression('(NIR - RED) / (NIR + RED)',
                          {'NIR': img.select('NIR'), 'RED': img.select('RED')}).rename('NDVI')
    ndwi = img.expression('(GREEN - NIR) / (GREEN + NIR)',
                          {'GREEN': img.select('GREEN'), 'NIR': img.select('NIR')}).rename('NDWI')
    ndmi = img.expression('(NIR - SWIR) / (NIR + SWIR)',
                          {'NIR': img.select('NIR'), 'SWIR': img.select('SWIR')}).rename('NDMI')
    savi = img.expression('1.2 * (NIR - RED) / (NIR + RED + 0.2)',
                          {'NIR': img.select('NIR'), 'RED': img.select('RED')}).rename('SAVI')
    return ee.Image.cat([ndvi, ndwi, ndmi, savi])

In [ ]:
def get_lst_features(year, region):
    """LST_summer/winter/annual + FDD + TDD, с gap-fill ERA5.
    ИСПРАВЛЕНО: фильтрация calendarRange ДО .map, чтобы не терять timestamps."""
    # Raw коллекции без преобразований (timestamps на месте)
    modis_raw = (ee.ImageCollection('MODIS/061/MOD11A1')
                 .filterDate(f'{year}-01-01', f'{year}-12-31')
                 .select('LST_Day_1km'))
    era5_raw = (ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR')
                .filterDate(f'{year}-01-01', f'{year}-12-31')
                .select('temperature_2m'))

    def to_celsius_modis(col):
        return col.mean().multiply(0.02).subtract(273.15)

    def to_celsius_era(col):
        return col.mean().subtract(273.15)

    def season_mean(months, name):
        m_modis = to_celsius_modis(
            modis_raw.filter(ee.Filter.calendarRange(months[0], months[1], 'month')))
        m_era = to_celsius_era(
            era5_raw.filter(ee.Filter.calendarRange(months[0], months[1], 'month')))
        return m_modis.unmask(m_era).rename(name)

    lst_summer = season_mean([6, 8], 'LST_summer')
    lst_winter = season_mean([12, 2], 'LST_winter')
    lst_annual = season_mean([1, 12], 'LST_annual')

    days_in_month = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
    fdd = ee.Image(0)
    tdd = ee.Image(0)
    for i, m in enumerate(range(1, 13)):
        mt = to_celsius_modis(
            modis_raw.filter(ee.Filter.calendarRange(m, m, 'month')))
        et = to_celsius_era(
            era5_raw.filter(ee.Filter.calendarRange(m, m, 'month')))
        t = mt.unmask(et)
        d = days_in_month[i]
        fdd = fdd.add(t.multiply(-1).max(0).multiply(d))
        tdd = tdd.add(t.max(0).multiply(d))

    return ee.Image.cat([lst_summer, lst_winter, lst_annual,
                         fdd.rename('FDD'), tdd.rename('TDD')])

In [ ]:
def get_snow(year):
    snow = (ee.ImageCollection('MODIS/061/MOD10A1')
            .filterDate(f'{year}-01-01', f'{year}-12-31')
            .select('NDSI_Snow_Cover'))
    return snow.map(lambda i: i.gt(40)).sum().rename('snow_days')

def get_terrain():
    dem = ee.Image('MERIT/DEM/v1_0_3').rename('elevation')
    slope = ee.Terrain.slope(dem).rename('slope')
    aspect = ee.Terrain.aspect(dem).rename('aspect')
    return ee.Image.cat([dem, slope, aspect])

def get_soil():
    oc = ee.Image('OpenLandMap/SOL/SOL_ORGANIC-CARBON_USDA-6A1C_M/v02').select('b0').rename('soil_oc')
    bd = ee.Image('OpenLandMap/SOL/SOL_BULKDENS-FINEEARTH_USDA-4A1H_M/v02').select('b0').rename('soil_bd')
    clay = ee.Image('OpenLandMap/SOL/SOL_CLAY-WFRACTION_USDA-3A1A1A_M/v02').select('b0').rename('soil_clay')
    return ee.Image.cat([oc, bd, clay])

def get_climate(year):
    wc = ee.Image('WORLDCLIM/V1/BIO')
    maat = wc.select('bio01').rename('MAAT')
    mAp = wc.select('bio12').rename('MAP')
    era5 = (ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR')
            .filterDate(f'{year}-01-01', f'{year}-12-31'))
    era5_t = era5.select('temperature_2m').mean().subtract(273.15).rename('era5_temp')
    era5_p = era5.select('total_precipitation_sum').mean().rename('era5_precip')
    return ee.Image.cat([maat, mAp, era5_t, era5_p])

## Главный цикл: 24 задачи (6 лет × 4 поддиапазона)

In [ ]:
def export_year_subband(year, band_name, west, east):
    region = ee.Geometry.Rectangle([west, SOUTH, east, NORTH], proj='EPSG:4326', geodesic=False)
    lons = ee.List.sequence(west, east, STEP_DEG)
    lats = ee.List.sequence(SOUTH, NORTH, STEP_DEG)
    def make_row(lat):
        lat = ee.Number(lat)
        return lons.map(lambda lon: ee.Feature(ee.Geometry.Point([ee.Number(lon), lat])))
    grid = ee.FeatureCollection(lats.map(make_row).flatten())

    img = ee.Image.cat([
        get_landsat(year, region),
        get_lst_features(year, region),
        get_snow(year),
        get_terrain(),
        get_soil(),
        get_climate(year)
    ])
    sampled = img.sampleRegions(collection=grid, scale=10000, geometries=True)
    fname = f'RussiaGrid_0.1deg_{year}_{band_name}'
    task = ee.batch.Export.table.toDrive(
        collection=sampled, description=fname, fileNamePrefix=fname,
        folder=EXPORT_FOLDER, fileFormat='GeoJSON')
    task.start()
    print(f'  Запущена: {fname}')
    return task

tasks = []
for year in YEARS:
    print(f'\n=== ГОД {year} ===')
    for band_name, w, e in BANDS:
        t = export_year_subband(year, band_name, w, e)
        tasks.append((year, band_name, t))

print(f'\nВсего запущено задач: {len(tasks)}')
print('Статусы: https://code.earthengine.google.com/tasks')

## Проверка статусов (запустить после ожидания)

In [ ]:
import time
from collections import Counter
states = Counter()
for year, band, t in tasks:
    s = t.status()
    states[s['state']] += 1
    if s['state'] == 'FAILED':
        print(f'  {year} {band}: FAILED — {s.get("error_message", "")}')
print('\nСтатистика:', dict(states))

## Что делать дальше

### Если задачи COMPLETED
1. Скачать все файлы из Google Drive (папка `GEE_exports_features_01deg/`)
2. Положить в `data/features_01deg/` локально
3. Сообщите мне — я подготовлю скрипт `merge_bands_01deg.py` для склейки полос band1_west+band1_east в band1

### Если есть FAILED по memory limit
- Если упал band0 или band2 — нужно его тоже разбить пополам
- Если упал band1_west или band1_east — придётся дробить ещё мельче (на четверти)
- Пришлите список упавших — починим

### Если FAILED по другой причине
- Пришлите текст ошибки — починю